# CSC14114 - Recommendation System with Implicit Feedback
**HCMUS - Đồ án Ứng dụng Dữ liệu Lớn**

| Model | Paper | Year |
|---|---|---|
| LightGCN | SIGIR 2020 | Graph CF baseline |
| SGL | SIGIR 2021 | + Edge Dropout + InfoNCE |
| SimGCL | SIGIR 2022 | + Noise Perturbation + InfoNCE |

**5 Datasets:** MovieLens-1M, Yelp2018, Amazon-Book, Gowalla, Steam

**Hướng dẫn:** Runtime → Change runtime type → **GPU T4** → Chạy từng cell từ trên xuống

In [ ]:
#@title **1. Setup**
import os, sys, json, time, random, zipfile, gzip, shutil, math
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import requests

plt.style.use('seaborn-v0_8-whitegrid')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
if DEVICE=='cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

BASE = Path('/content/recsys')
RAW = BASE/'raw'; PROC = BASE/'processed'; RES = BASE/'results'
for d in [RAW, PROC, RES]: d.mkdir(parents=True, exist_ok=True)

---
# Phase 2: Download & Preprocess Datasets

In [ ]:
#@title **2. Utility functions**

def download(url, dest, desc=''):
    if dest.exists(): print(f'  {dest.name} exists, skip'); return
    print(f'  Downloading {desc}...')
    r = requests.get(url, stream=True, timeout=300); r.raise_for_status()
    total = int(r.headers.get('content-length',0))
    with open(dest,'wb') as f, tqdm(total=total,unit='B',unit_scale=True,desc=desc) as bar:
        for chunk in r.iter_content(65536): f.write(chunk); bar.update(len(chunk))

def k_core(df, k, uc='user_id', ic='item_id'):
    while True:
        n = len(df)
        df = df[df[uc].isin(df[uc].value_counts()[lambda x:x>=k].index)]
        df = df[df[ic].isin(df[ic].value_counts()[lambda x:x>=k].index)]
        if len(df)==n: break
    return df

def process_save(df, name, kc=10):
    print(f'\n--- {name} ---')
    print(f'  Raw: {len(df):,} rows, {df.user_id.nunique()} users, {df.item_id.nunique()} items')
    df = df.drop_duplicates(['user_id','item_id'], keep='first')
    df = k_core(df, kc)
    users = sorted(df.user_id.unique()); items = sorted(df.item_id.unique())
    u2i = {u:i for i,u in enumerate(users)}; i2i = {it:i for i,it in enumerate(items)}
    df = df.copy()
    df['uid'] = df.user_id.map(u2i); df['iid'] = df.item_id.map(i2i)
    nu, ni = len(users), len(items)
    if 'ts' in df.columns and df.ts.sum()>0: df = df.sort_values(['uid','ts'])
    else: df = df.sort_values('uid')
    train, val, test = [], [], []
    for uid, g in df.groupby('uid'):
        its = g.iid.tolist()
        if len(its)<3: train+=[(uid,x) for x in its]
        else: test.append((uid,its[-1])); val.append((uid,its[-2])); train+=[(uid,x) for x in its[:-2]]
    tr=pd.DataFrame(train,columns=['uid','iid'])
    va=pd.DataFrame(val,columns=['uid','iid'])
    te=pd.DataFrame(test,columns=['uid','iid'])
    out=PROC/name; out.mkdir(exist_ok=True)
    tr.to_csv(out/'train.csv',index=False); va.to_csv(out/'val.csv',index=False); te.to_csv(out/'test.csv',index=False)
    info={'name':name,'n_users':nu,'n_items':ni,'n_inter':len(df),'density':len(df)/(nu*ni),
          'n_train':len(tr),'n_val':len(va),'n_test':len(te)}
    json.dump(info,open(out/'info.json','w'),indent=2)
    print(f'  Final: {nu:,} users, {ni:,} items, {len(df):,} inter, density={len(df)/(nu*ni)*100:.4f}%')
    print(f'  Split: train={len(tr):,} val={len(va):,} test={len(te):,}')
    return info

def load_lgcn_txt(d):
    rows=[]
    for fn in ['train.txt','test.txt']:
        fp=d/fn
        if not fp.exists(): continue
        for line in open(fp):
            p=line.strip().split()
            if len(p)<2: continue
            uid=int(p[0])
            for iid in p[1:]: rows.append({'user_id':uid,'item_id':int(iid),'ts':0})
    return pd.DataFrame(rows)

print('Utilities ready')

In [ ]:
#@title **3. Download & Process MovieLens-1M**
ml_zip = RAW/'ml-1m.zip'
download('https://files.grouplens.org/datasets/movielens/ml-1m.zip', ml_zip, 'ML-1M')
if not (RAW/'ml-1m').exists():
    with zipfile.ZipFile(ml_zip) as z: z.extractall(RAW)
df = pd.read_csv(RAW/'ml-1m'/'ratings.dat', sep='::', header=None,
                 names=['user_id','item_id','rating','ts'], engine='python', encoding='latin-1')
info_ml = process_save(df, 'ml-1m', kc=5)

In [ ]:
#@title **4. Download & Process Gowalla**
gd = RAW/'gowalla'; gd.mkdir(exist_ok=True)
gz = gd/'checkins.txt.gz'; txt = gd/'checkins.txt'
download('https://snap.stanford.edu/data/loc-gowalla_totalCheckins.txt.gz', gz, 'Gowalla')
if not txt.exists():
    with gzip.open(gz,'rb') as fi, open(txt,'wb') as fo: shutil.copyfileobj(fi,fo)
df = pd.read_csv(txt, sep='\t', header=None, names=['user_id','ts_str','lat','lon','item_id'])
df['ts'] = pd.to_datetime(df.ts_str, errors='coerce').astype(np.int64)//10**9
df = df[['user_id','item_id','ts']]
info_gow = process_save(df, 'gowalla', kc=10)

In [ ]:
#@title **5. Download & Process Yelp2018**
yd = RAW/'yelp2018'; yd.mkdir(exist_ok=True)
base = 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018'
for fn in ['train.txt','test.txt']: download(f'{base}/{fn}', yd/fn, f'Yelp {fn}')
df = load_lgcn_txt(yd)
info_yelp = process_save(df, 'yelp2018', kc=5)

In [ ]:
#@title **6. Download & Process Amazon-Book**
ad = RAW/'amazon-book'; ad.mkdir(exist_ok=True)
base = 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book'
for fn in ['train.txt','test.txt']: download(f'{base}/{fn}', ad/fn, f'AmazonBook {fn}')
df = load_lgcn_txt(ad)
info_ab = process_save(df, 'amazon-book', kc=5)

In [ ]:
#@title **7. Thu thập Steam Dataset (tự collect qua API)**
# Thu thập implicit feedback từ Steam Store Reviews API
# Mỗi review = user đã mua và chơi game => implicit interaction

steam_csv = RAW/'steam_interactions.csv'

if not steam_csv.exists():
    print('=== Thu thập Steam data qua Reviews API ===')

    # Lấy danh sách games
    try:
        resp = requests.get('https://api.steampowered.com/ISteamApps/GetAppList/v2/', timeout=30)
        apps = [a['appid'] for a in resp.json()['applist']['apps'] if a.get('name','').strip()]
        random.seed(42); random.shuffle(apps)
        app_ids = apps[:1500]
        print(f'Got {len(apps)} apps, sampling {len(app_ids)}')
    except:
        print('API list failed, using known popular app IDs')
        app_ids = [730,570,440,4000,550,10,240,400,320,380,340,500,360,300,220,
            620,72850,230410,271590,359550,292030,236390,252490,304930,227300,
            8930,48700,105600,218620,238960,242760,49520,206420,212680,203160,
            250900,322330,242920,242050,261550,201810,214490,274170,265930,
            289070,301520,310380,322170,346110,349040,359050,367520,377160,
            394360,413150,427520,440900,460790,493520,524220,550650,578080,
            582010,611670,620980,648800,728880,813780,892970,945360,1086940,
            1091500,1172470,1174180,1245620,1326470,1599340,1716740,2050650,
            252950,286160,248390,283640,233450,233270,250760,211820,239140,
            251570,245170,248820,234140,236430,249050,227380,233720,222880,
            212480,219640,209000,207610,204360,203770,200710,57690,55230,
            41060,38410,38600,35450,34270,28050,25800,22380,22330,22200,
            21090,20920,20510,20200,19900,17460,15620,15520,15400,14100,
            13230,13200,12900,12830,12500,12210,11360,10180,9480,9420,
            8190,7670,6860,6520,6510,4920,4560,4540,4500,3920,3900,3480,
            2600,2400,2300,2200,2100,1520,1510,1500,1250,100,70,60,50,
            107410,113020,200510,201790,204100,208580,224260,224580,228980,
            231430,233450,234650,237110,239030,239820,242050,244210,244850,
            246620,247870,249590,251150,255220,257510,259080,263280,266840,
            269950,275850,279720,281990,282070,282440,289130,294100,298110,
            300380,302510,306130,311210,312530,313120,314660,317400,319630,
            341800,348250,354380,361420,362490,365450,374320,378420,379430,
            381210,386360,388880,392160,397540,418370,431960,433340,438100,
            444090,460950,462770,466560,475150,489830,504230,513710,548430,
            553850,570940,588430,594650,632360,678950,812140,814380,863550,
            952060,1145360,1238810,1422450,1449850,1486860,1794680,2358720]

    all_inter = []
    for app_id in tqdm(app_ids, desc='Collecting Steam reviews'):
        cursor = '*'
        got = 0
        while got < 2000:
            try:
                r = requests.get(f'https://store.steampowered.com/appreviews/{app_id}',
                    params={'json':1,'filter':'all','language':'all','num_per_page':100,
                            'cursor':cursor,'purchase_type':'all'}, timeout=10)
                data = r.json()
            except: break
            if not data.get('success'): break
            reviews = data.get('reviews',[])
            if not reviews: break
            for rv in reviews:
                a = rv.get('author',{})
                uid = a.get('steamid'); pt = a.get('playtime_forever',0)
                if uid and pt>0:
                    all_inter.append({'user_id':uid,'item_id':app_id,'ts':rv.get('timestamp_created',0)})
            got += len(reviews)
            cursor = data.get('cursor','')
            if not cursor or len(reviews)<100: break
            time.sleep(0.25)
        if len(all_inter)>=1_200_000:
            print(f'\nReached {len(all_inter):,} interactions'); break
        time.sleep(0.15)

    df_steam = pd.DataFrame(all_inter)
    df_steam.to_csv(steam_csv, index=False)
    print(f'Saved {len(df_steam):,} interactions')
else:
    df_steam = pd.read_csv(steam_csv)
    print(f'Loaded {len(df_steam):,} cached interactions')

print(f'Steam raw: {len(df_steam):,} inter, {df_steam.user_id.nunique():,} users, {df_steam.item_id.nunique():,} items')

# Xử lý: nếu ít hơn 100k thì k_core nhỏ hơn
kc_steam = 3 if len(df_steam) < 200000 else (5 if len(df_steam) < 500000 else 10)
info_steam = process_save(df_steam, 'steam', kc=kc_steam)

In [ ]:
#@title **8. Thống kê tổng hợp datasets**
all_info = []
DATASETS = []
for ds in ['ml-1m','gowalla','yelp2018','amazon-book','steam']:
    ip = PROC/ds/'info.json'
    if ip.exists():
        info = json.load(open(ip))
        all_info.append(info)
        DATASETS.append(ds)

df_info = pd.DataFrame(all_info)
print(df_info[['name','n_users','n_items','n_inter','density','n_train','n_val','n_test']].to_string(index=False))

fig, axes = plt.subplots(1,3,figsize=(18,5))
colors = sns.color_palette('husl', len(df_info))
for ax,col,title in zip(axes,['n_users','n_items','n_inter'],['Users','Items','Interactions']):
    bars = ax.bar(df_info['name'], df_info[col], color=colors)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    for b,v in zip(bars, df_info[col]):
        ax.text(b.get_x()+b.get_width()/2, b.get_height(), f'{v:,.0f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig(RES/'dataset_stats.png',dpi=150,bbox_inches='tight'); plt.show()
print(f'\nAvailable datasets: {DATASETS}')

---
# Phase 3: Models, Training & Evaluation

In [ ]:
#@title **9. Define Models: LightGCN, SGL, SimGCL**

def build_adj(nu, ni, u_arr, i_arr, device):
    n = nu + ni
    row = np.concatenate([u_arr, i_arr+nu])
    col = np.concatenate([i_arr+nu, u_arr])
    adj = sp.csr_matrix((np.ones(len(row),dtype=np.float32),(row,col)), shape=(n,n))
    deg = np.array(adj.sum(1)).flatten()
    d_inv = np.power(deg+1e-10,-0.5); d_inv[np.isinf(d_inv)]=0.
    norm = (sp.diags(d_inv) @ adj @ sp.diags(d_inv)).tocoo()
    return torch.sparse_coo_tensor(
        torch.LongTensor(np.array([norm.row,norm.col])),
        torch.FloatTensor(norm.data), (n,n)).to(device)

class LightGCN(nn.Module):
    def __init__(s, nu, ni, dim=64, L=3, reg=1e-4):
        super().__init__()
        s.nu,s.ni,s.L,s.reg = nu,ni,L,reg
        s.ue = nn.Embedding(nu,dim); s.ie = nn.Embedding(ni,dim)
        nn.init.xavier_uniform_(s.ue.weight); nn.init.xavier_uniform_(s.ie.weight)
        s.adj = None
    def prop(s, adj=None):
        if adj is None: adj=s.adj
        e = torch.cat([s.ue.weight, s.ie.weight])
        out=[e]
        for _ in range(s.L): e=torch.sparse.mm(adj,e); out.append(e)
        f = torch.stack(out,1).mean(1)
        return f[:s.nu], f[s.nu:]
    def bpr(s, u, p, n):
        ue,ie = s.prop()
        eu,ep,en = ue[u],ie[p],ie[n]
        loss = -F.logsigmoid((eu*ep).sum(-1)-(eu*en).sum(-1)).mean()
        reg = s.reg*(s.ue.weight[u].norm(2).pow(2)+s.ie.weight[p].norm(2).pow(2)+s.ie.weight[n].norm(2).pow(2))/len(u)
        return loss+reg
    def scores(s, u):
        ue,ie = s.prop()
        return ue[u] @ ie.T

class SGL(nn.Module):
    def __init__(s, nu, ni, dim=64, L=3, reg=1e-4, drop=0.1, tau=0.2, lam=0.1):
        super().__init__()
        s.nu,s.ni,s.L,s.reg = nu,ni,L,reg
        s.drop,s.tau,s.lam = drop,tau,lam
        s.ue = nn.Embedding(nu,dim); s.ie = nn.Embedding(ni,dim)
        nn.init.xavier_uniform_(s.ue.weight); nn.init.xavier_uniform_(s.ie.weight)
        s.adj = None; s._tu=s._ti=s._dev=None
    def prop(s, adj=None):
        if adj is None: adj=s.adj
        e = torch.cat([s.ue.weight, s.ie.weight])
        out=[e]
        for _ in range(s.L): e=torch.sparse.mm(adj,e); out.append(e)
        f = torch.stack(out,1).mean(1)
        return f[:s.nu], f[s.nu:]
    def _aug(s):
        mask = np.random.rand(len(s._tu))>s.drop
        return build_adj(s.nu,s.ni,s._tu[mask],s._ti[mask],s._dev)
    def bpr(s, u, p, n):
        ue,ie = s.prop()
        eu,ep,en = ue[u],ie[p],ie[n]
        loss = -F.logsigmoid((eu*ep).sum(-1)-(eu*en).sum(-1)).mean()
        reg = s.reg*(s.ue.weight[u].norm(2).pow(2)+s.ie.weight[p].norm(2).pow(2)+s.ie.weight[n].norm(2).pow(2))/len(u)
        a1,a2 = s._aug(), s._aug()
        ue1,ie1 = s.prop(a1); ue2,ie2 = s.prop(a2)
        uu,ii = torch.unique(u), torch.unique(p)
        z1=F.normalize(ue1[uu],-1); z2=F.normalize(ue2[uu],-1)
        ssl_u = F.cross_entropy(z1@z2.T/s.tau, torch.arange(len(uu),device=u.device))
        z1=F.normalize(ie1[ii],-1); z2=F.normalize(ie2[ii],-1)
        ssl_i = F.cross_entropy(z1@z2.T/s.tau, torch.arange(len(ii),device=u.device))
        return loss+reg+s.lam*(ssl_u+ssl_i)
    def scores(s, u):
        ue,ie = s.prop()
        return ue[u] @ ie.T

class SimGCL(nn.Module):
    def __init__(s, nu, ni, dim=64, L=3, reg=1e-4, eps=0.1, tau=0.2, lam=0.1):
        super().__init__()
        s.nu,s.ni,s.L,s.reg = nu,ni,L,reg
        s.eps,s.tau,s.lam = eps,tau,lam
        s.ue = nn.Embedding(nu,dim); s.ie = nn.Embedding(ni,dim)
        nn.init.xavier_uniform_(s.ue.weight); nn.init.xavier_uniform_(s.ie.weight)
        s.adj = None
    def prop(s, noise=False):
        e = torch.cat([s.ue.weight, s.ie.weight])
        out=[e]
        for _ in range(s.L):
            e = torch.sparse.mm(s.adj, e)
            if noise: e = e + F.normalize(torch.rand_like(e)*2-1,-1)*s.eps
            out.append(e)
        f = torch.stack(out,1).mean(1)
        return f[:s.nu], f[s.nu:]
    def bpr(s, u, p, n):
        ue,ie = s.prop()
        eu,ep,en = ue[u],ie[p],ie[n]
        loss = -F.logsigmoid((eu*ep).sum(-1)-(eu*en).sum(-1)).mean()
        reg = s.reg*(s.ue.weight[u].norm(2).pow(2)+s.ie.weight[p].norm(2).pow(2)+s.ie.weight[n].norm(2).pow(2))/len(u)
        ue1,ie1 = s.prop(noise=True); ue2,ie2 = s.prop(noise=True)
        uu,ii = torch.unique(u), torch.unique(p)
        z1=F.normalize(ue1[uu],-1); z2=F.normalize(ue2[uu],-1)
        ssl_u = F.cross_entropy(z1@z2.T/s.tau, torch.arange(len(uu),device=u.device))
        z1=F.normalize(ie1[ii],-1); z2=F.normalize(ie2[ii],-1)
        ssl_i = F.cross_entropy(z1@z2.T/s.tau, torch.arange(len(ii),device=u.device))
        return loss+reg+s.lam*(ssl_u+ssl_i)
    def scores(s, u):
        ue,ie = s.prop()
        return ue[u] @ ie.T

print('Models: LightGCN, SGL, SimGCL defined')

In [ ]:
#@title **10. Evaluation Metrics & Training Loop**

def calc_metrics(topk, truth, ks=[10,20]):
    res = {f'{m}@{k}':[] for k in ks for m in ['recall','ndcg','hr','prec']}
    for i in range(len(topk)):
        rec = topk[i]
        gt = set(truth[i]) if isinstance(truth[i],list) else {truth[i]}
        for k in ks:
            rk = rec[:k]; hits = sum(1 for r in rk if r in gt)
            res[f'prec@{k}'].append(hits/k)
            res[f'recall@{k}'].append(hits/max(len(gt),1))
            res[f'hr@{k}'].append(1. if hits>0 else 0.)
            dcg = sum(1./np.log2(j+2) for j,r in enumerate(rk) if r in gt)
            idcg = sum(1./np.log2(j+2) for j in range(min(len(gt),k)))
            res[f'ndcg@{k}'].append(dcg/idcg if idcg>0 else 0.)
    return {k:np.mean(v) for k,v in res.items()}

@torch.no_grad()
def evaluate(model, test_df, train_d, ni, ks=[10,20], bs=1024):
    model.eval()
    mk = max(ks)
    td = defaultdict(list)
    for _,r in test_df.iterrows(): td[int(r.uid)].append(int(r.iid))
    users = sorted(td.keys())
    all_tk, all_gt = [], []
    dev = next(model.parameters()).device
    for i in range(0,len(users),bs):
        batch = users[i:i+bs]
        ut = torch.LongTensor(batch).to(dev)
        sc = model.scores(ut)
        for j,uid in enumerate(batch):
            if uid in train_d: sc[j, list(train_d[uid])] = -1e9
        _, tk = sc.topk(mk,-1)
        tk = tk.cpu().numpy()
        for j,uid in enumerate(batch): all_tk.append(tk[j]); all_gt.append(td[uid])
    return calc_metrics(all_tk, all_gt, ks)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def fast_neg(users, ni, train_d):
    neg = np.random.randint(0,ni,len(users))
    for j in range(len(users)):
        while neg[j] in train_d.get(users[j], set()): neg[j]=random.randint(0,ni-1)
    return neg

def train_one(model_name, ds_name, device=DEVICE, seed=42,
              dim=64, L=3, lr=1e-3, bs=2048, epochs=80, patience=15):
    set_seed(seed)
    dp = PROC/ds_name
    if not dp.exists(): print(f'{ds_name} not found'); return None
    tr = pd.read_csv(dp/'train.csv')
    va = pd.read_csv(dp/'val.csv')
    te = pd.read_csv(dp/'test.csv')
    info = json.load(open(dp/'info.json'))
    nu, ni = info['n_users'], info['n_items']
    train_d = defaultdict(set)
    for _,r in tr.iterrows(): train_d[int(r.uid)].add(int(r.iid))
    tu = tr.uid.values.astype(np.int64)
    ti = tr.iid.values.astype(np.int64)
    adj = build_adj(nu,ni,tu,ti,device)

    if model_name=='lightgcn': model = LightGCN(nu,ni,dim,L).to(device)
    elif model_name=='sgl':
        model = SGL(nu,ni,dim,L).to(device)
        model._tu,model._ti,model._dev = tu,ti,device
    elif model_name=='simgcl': model = SimGCL(nu,ni,dim,L).to(device)
    model.adj = adj
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_ndcg, best_ep, wait, best_st = 0.,0,0,None

    print(f'\n{"="*60}')
    print(f'  {model_name.upper()} on {ds_name} | seed={seed} | {nu} users {ni} items {len(tr):,} train')
    print(f'{"="*60}')
    t0 = time.time()

    for ep in range(1, epochs+1):
        model.train()
        idx = np.random.permutation(len(tu)); ep_loss=0.; nb=0
        for s in range(0,len(tu),bs):
            bi = idx[s:min(s+bs,len(tu))]
            bu,bp = tu[bi], ti[bi]
            bn = fast_neg(bu,ni,train_d)
            opt.zero_grad()
            loss = model.bpr(torch.LongTensor(bu).to(device),
                             torch.LongTensor(bp).to(device),
                             torch.LongTensor(bn).to(device))
            loss.backward(); opt.step()
            ep_loss+=loss.item(); nb+=1

        if ep%5==0 or ep==1:
            vm = evaluate(model, va, train_d, ni)
            vn = vm['ndcg@20']; dt = time.time()-t0
            print(f'  Ep {ep:3d} | loss={ep_loss/nb:.4f} | ndcg@20={vn:.4f} recall@20={vm["recall@20"]:.4f} | {dt:.0f}s')
            if vn>best_ndcg:
                best_ndcg,best_ep,wait = vn,ep,0
                best_st = {k:v.cpu().clone() for k,v in model.state_dict().items()}
            else:
                wait+=1
                if wait>=patience//5: print(f'  Early stop ep {ep} (best {best_ep})'); break

    if best_st: model.load_state_dict({k:v.to(device) for k,v in best_st.items()})
    tm = evaluate(model, te, train_d, ni)
    tt = time.time()-t0
    print(f'  TEST: '+' | '.join(f'{k}={v:.4f}' for k,v in sorted(tm.items())))
    print(f'  time={tt:.1f}s best_ep={best_ep}')

    result = {'model':model_name,'dataset':ds_name,'seed':seed,'best_epoch':best_ep,'time_sec':round(tt,1),**tm}
    rd = RES/model_name; rd.mkdir(parents=True, exist_ok=True)
    json.dump(result,open(rd/f'{ds_name}_s{seed}.json','w'),indent=2)
    return result

print('Training & evaluation functions ready')

In [ ]:
#@title **11. RUN ALL 15 EXPERIMENTS** (3 models x 5 datasets x 3 seeds)
# ~2-4 giờ với GPU T4

MODELS = ['lightgcn', 'sgl', 'simgcl']
SEEDS = [42, 123, 456]

print(f'Datasets: {DATASETS}')
print(f'Models: {MODELS}')
print(f'Seeds: {SEEDS}')
print(f'Total runs: {len(MODELS)*len(DATASETS)*len(SEEDS)}')

all_results = []
t_start = time.time()

for m in MODELS:
    for d in DATASETS:
        for seed in SEEDS:
            try:
                r = train_one(m, d, device=DEVICE, seed=seed, epochs=80, patience=15)
                if r: all_results.append(r)
            except Exception as ex:
                print(f'ERROR {m}/{d}: {ex}')
                import traceback; traceback.print_exc()

total_min = (time.time()-t_start)/60
print(f'\n\nALL DONE: {len(all_results)} runs in {total_min:.1f} min')

In [ ]:
#@title **12. Results Summary Table (mean ± std)**

df_all = pd.DataFrame(all_results)
mcols = [c for c in df_all.columns if '@' in c]

rows = []
for (m,d), g in df_all.groupby(['model','dataset']):
    row = {'model':m, 'dataset':d, 'runs':len(g), 'avg_time':g.time_sec.mean()}
    for mc in mcols: row[f'{mc}_mean']=g[mc].mean(); row[f'{mc}_std']=g[mc].std()
    rows.append(row)

df_sum = pd.DataFrame(rows)
df_sum.to_csv(RES/'summary.csv', index=False)

print('\n'+'='*90)
print('FINAL RESULTS')
print('='*90)
for metric in ['ndcg@20','recall@20','hr@20','prec@20']:
    print(f'\n{metric.upper()}:')
    hdr = f'{"Model":<12}'+"".join(f'{d:<20}' for d in DATASETS)
    print(hdr); print('-'*len(hdr))
    for m in MODELS:
        row = f'{m:<12}'
        for d in DATASETS:
            match = df_sum[(df_sum.model==m)&(df_sum.dataset==d)]
            if len(match):
                mn = match[f'{metric}_mean'].values[0]
                sd = match[f'{metric}_std'].values[0]
                row += f'{mn:.4f}\u00b1{sd:.4f}      '
            else: row += f'{"N/A":<20}'
        print(row)

In [ ]:
#@title **13. Visualization: Model Comparison Bar Charts**

COLORS = {'lightgcn':'#2196F3','sgl':'#FF9800','simgcl':'#4CAF50'}

fig, axes = plt.subplots(2,2,figsize=(18,12))
for ax, metric in zip(axes.flat, ['ndcg@20','recall@20','hr@20','ndcg@10']):
    x = np.arange(len(DATASETS)); w = 0.25
    for i,m in enumerate(MODELS):
        vals = [df_sum[(df_sum.model==m)&(df_sum.dataset==d)][f'{metric}_mean'].values[0]
                if len(df_sum[(df_sum.model==m)&(df_sum.dataset==d)]) else 0 for d in DATASETS]
        errs = [df_sum[(df_sum.model==m)&(df_sum.dataset==d)][f'{metric}_std'].values[0]
                if len(df_sum[(df_sum.model==m)&(df_sum.dataset==d)]) else 0 for d in DATASETS]
        ax.bar(x+i*w, vals, w, yerr=errs, label=m.upper(), color=COLORS[m], alpha=0.85, capsize=3)
    ax.set_xticks(x+w); ax.set_xticklabels(DATASETS, rotation=15)
    ax.set_ylabel(metric.upper()); ax.set_title(metric.upper(), fontsize=14, fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.suptitle('Model Comparison Across Datasets', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig(RES/'comparison.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
#@title **14. Improvement Heatmap (SGL/SimGCL vs LightGCN)**

fig, axes = plt.subplots(1,2,figsize=(16,5))
for ax, m in zip(axes, ['sgl','simgcl']):
    imp = {}
    for metric in ['ndcg@10','ndcg@20','recall@10','recall@20','hr@10','hr@20']:
        vals = []
        for d in DATASETS:
            b = df_sum[(df_sum.model=='lightgcn')&(df_sum.dataset==d)]
            c = df_sum[(df_sum.model==m)&(df_sum.dataset==d)]
            if len(b) and len(c):
                bv = b[f'{metric}_mean'].values[0]
                cv = c[f'{metric}_mean'].values[0]
                vals.append((cv-bv)/bv*100 if bv>0 else 0)
            else: vals.append(0)
        imp[metric] = vals
    df_imp = pd.DataFrame(imp, index=DATASETS)
    sns.heatmap(df_imp, annot=True, fmt='.1f', cmap='RdYlGn', center=0, ax=ax, linewidths=.5)
    ax.set_title(f'{m.upper()} vs LightGCN (% Improvement)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(RES/'improvement.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
#@title **15. Density vs Performance & Training Time**

ds_info = {d: json.load(open(PROC/d/'info.json')) for d in DATASETS if (PROC/d/'info.json').exists()}

fig, axes = plt.subplots(1,3,figsize=(20,5))

# Density vs NDCG
for m in MODELS:
    dens, perf = [], []
    for d in DATASETS:
        match = df_sum[(df_sum.model==m)&(df_sum.dataset==d)]
        if len(match) and d in ds_info:
            dens.append(ds_info[d]['density']*100); perf.append(match['ndcg@20_mean'].values[0])
    axes[0].scatter(dens,perf,label=m.upper(),color=COLORS[m],s=100)
axes[0].set_xlabel('Density (%)'); axes[0].set_ylabel('NDCG@20')
axes[0].set_title('Density vs NDCG@20',fontweight='bold'); axes[0].legend()

# Density vs Recall
for m in MODELS:
    dens, perf = [], []
    for d in DATASETS:
        match = df_sum[(df_sum.model==m)&(df_sum.dataset==d)]
        if len(match) and d in ds_info:
            dens.append(ds_info[d]['density']*100); perf.append(match['recall@20_mean'].values[0])
    axes[1].scatter(dens,perf,label=m.upper(),color=COLORS[m],s=100)
axes[1].set_xlabel('Density (%)'); axes[1].set_ylabel('Recall@20')
axes[1].set_title('Density vs Recall@20',fontweight='bold'); axes[1].legend()

# Training time
x = np.arange(len(DATASETS)); w=0.25
for i,m in enumerate(MODELS):
    t = [df_sum[(df_sum.model==m)&(df_sum.dataset==d)].avg_time.values[0]
         if len(df_sum[(df_sum.model==m)&(df_sum.dataset==d)]) else 0 for d in DATASETS]
    axes[2].bar(x+i*w,t,w,label=m.upper(),color=COLORS[m],alpha=0.85)
axes[2].set_xticks(x+w); axes[2].set_xticklabels(DATASETS,rotation=15)
axes[2].set_ylabel('Seconds'); axes[2].set_title('Training Time',fontweight='bold'); axes[2].legend()

plt.tight_layout(); plt.savefig(RES/'analysis.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
#@title **16. LaTeX Tables cho báo cáo**

for metric in ['ndcg@20','recall@20','hr@20']:
    print(f'\n% --- {metric.upper()} ---')
    print(r'\begin{table}[h]\centering')
    print(f'\\caption{{Performance ({metric.upper()})}}')
    print(r'\begin{tabular}{l'+'c'*len(DATASETS)+'}')
    print(r'\hline')
    print('Model & '+' & '.join(DATASETS)+r' \\')
    print(r'\hline')
    for m in MODELS:
        row = m.upper()
        for d in DATASETS:
            match = df_sum[(df_sum.model==m)&(df_sum.dataset==d)]
            if len(match):
                mn = match[f'{metric}_mean'].values[0]
                sd = match[f'{metric}_std'].values[0]
                row += f' & {mn:.4f}$\\pm${sd:.4f}'
            else: row += ' & -'
        print(row+r' \\')
    print(r'\hline')
    print(r'\end{tabular}\end{table}')

In [ ]:
#@title **17. Download kết quả về máy**

shutil.make_archive('/content/recsys_results', 'zip', RES)
try:
    from google.colab import files
    files.download('/content/recsys_results.zip')
    print('Downloading...')
except:
    print(f'Results at: {RES}')
    for f in sorted(RES.rglob('*')):
        if f.is_file(): print(f'  {f.relative_to(RES)}')